# 04 — Training

Train the transformer to predict the next token in Gambler's Ruin sequences
via cross-entropy loss.  Save the best checkpoint to disk.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

DATA_DIR = Path("projects/markov-transformer/experiments/markov-chain-learning/data")
print("Data dir:", DATA_DIR.resolve())

## Configuration

In [ ]:
# Model
VOCAB_SIZE = 5
D_MODEL = 32
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
MAX_LEN = 64

# Training
BATCH_SIZE = 256
EPOCHS = 50
LR = 3e-3
VAL_SPLIT = 0.1  # 10% held-out validation
SEED = 42
torch.manual_seed(SEED)

## Model (copy from 03_model.ipynb for self-containment)

In [ ]:
class MarkovTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=5,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        max_len=64,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L = x.shape
        positions = torch.arange(L, device=x.device)
        h = self.token_embedding(x) + self.pos_embedding(positions)
        mask = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device), diagonal=1
        )
        h = self.transformer(h, mask=mask, is_causal=True)
        return self.head(h)

## Load data

In [ ]:
class MarkovSequenceDataset(Dataset):
    def __init__(self, sequences: torch.Tensor):
        self.data = sequences

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


checkpoint = torch.load(DATA_DIR / "sequences.pt", weights_only=True)
sequences = checkpoint["sequences"]
T_true = checkpoint["T"]
print(f"Loaded {len(sequences)} sequences of length {sequences.shape[1]}")

full_ds = MarkovSequenceDataset(sequences)
n_val = int(len(full_ds) * VAL_SPLIT)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {n_train}   Val: {n_val}")

## Training loop

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

model = MarkovTransformer(
    VOCAB_SIZE, D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD, MAX_LEN
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()


def run_epoch(loader, training: bool):
    model.train(training)
    total_loss, n_tokens = 0.0, 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = batch.to(device)  # (B, L)
            x_in = batch[:, :-1]  # context  (B, L-1)
            x_tgt = batch[:, 1:]  # targets  (B, L-1)

            logits = model(x_in)  # (B, L-1, V)
            B, Lm1, V = logits.shape
            loss = criterion(logits.reshape(-1, V), x_tgt.reshape(-1))

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * B * Lm1
            n_tokens += B * Lm1

    return total_loss / n_tokens


train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in tqdm(range(1, EPOCHS + 1), desc="Training"):
    tr_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)
    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "model_state": model.state_dict(),
                "config": dict(
                    vocab_size=VOCAB_SIZE,
                    d_model=D_MODEL,
                    nhead=NHEAD,
                    num_layers=NUM_LAYERS,
                    dim_feedforward=DIM_FEEDFORWARD,
                    max_len=MAX_LEN,
                ),
                "best_val_loss": best_val_loss,
                "epoch": epoch,
            },
            DATA_DIR / "checkpoint.pt",
        )

print(f"\nBest val loss: {best_val_loss:.4f}  (epoch saved to checkpoint.pt)")
print(f"Entropy of uniform-over-5: {torch.log(torch.tensor(5.0)):.4f}  (upper bound)")

## Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Val", linestyle="--")
ax.axhline(
    torch.log(torch.tensor(5.0)).item(),
    color="gray",
    linestyle=":",
    linewidth=0.8,
    label="Entropy(Uniform[5])",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss (per token)")
ax.set_title("Training curve")
ax.legend()
plt.tight_layout()
plt.show()